Imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

Load Dataset

In [2]:
df = pd.read_csv("../data/dataset.csv")

print("Total rows:", len(df))
df.head()

Total rows: 600


,num_loops,max_loop_depth,num_if_statements,num_return_statements,assignment_operations,augmented_assignments,comparison_operations,has_recursion,recursive_call_count,recursion_with_loop,...,nested_subscript_usage,dict_update_count,membership_checks,uses_sorted,uses_range,uses_enumerate,lines_of_code,num_functions,pattern_label,efficiency_label
0,2,2,1,2,0,0,1,0,0,0,...,0,0,0,0,0,0,6,1,brute_force,efficient
1,2,2,2,2,1,0,2,0,0,0,...,0,0,0,0,1,0,8,1,brute_force,efficient
2,2,2,1,2,0,0,1,0,0,0,...,0,0,0,0,1,0,6,1,brute_force,efficient
3,2,2,2,2,0,0,3,0,0,0,...,0,0,0,0,0,1,7,1,brute_force,efficient
4,2,2,2,1,3,0,2,0,0,0,...,0,0,0,0,1,0,9,1,brute_force,efficient


Check Class Distribution

In [3]:
print("Pattern Distribution:")
print(df["pattern_label"].value_counts())

print("\nEfficiency Distribution:")
print(df["efficiency_label"].value_counts())

Pattern Distribution:
pattern_label
brute_force            100
dynamic_programming    100
hashing                100
recursion              100
stack                  100
two_pointer            100
Name: count, dtype: int64

Efficiency Distribution:
efficiency_label
suboptimal     205
inefficient    200
efficient      195
Name: count, dtype: int64


Separate Features and Labels

In [4]:
X = df.drop(["pattern_label", "efficiency_label"], axis=1)
feature_columns = X.columns.tolist()

import joblib
joblib.dump(feature_columns, "backend/models/feature_columns.pkl")

print("Saved feature columns:", feature_columns)

pattern_encoder = LabelEncoder()
eff_encoder = LabelEncoder()

y_pattern = pattern_encoder.fit_transform(df["pattern_label"])
y_eff = eff_encoder.fit_transform(df["efficiency_label"])

Saved feature columns: ['num_loops', 'max_loop_depth', 'num_if_statements', 'num_return_statements', 'assignment_operations', 'augmented_assignments', 'comparison_operations', 'has_recursion', 'recursive_call_count', 'recursion_with_loop', 'uses_list', 'uses_dict', 'uses_set', 'uses_append', 'uses_pop', 'uses_2d_list', 'uses_subscript_assignment', 'dict_subscript_usage', 'nested_subscript_usage', 'dict_update_count', 'membership_checks', 'uses_sorted', 'uses_range', 'uses_enumerate', 'lines_of_code', 'num_functions']


Train/Test Split

In [5]:
X_train, X_test, y_pattern_train, y_pattern_test = train_test_split(
    X,
    y_pattern,
    test_size=0.2,
    random_state=42,
    stratify=y_pattern
)

_, _, y_eff_train, y_eff_test = train_test_split(
    X,
    y_eff,
    test_size=0.2,
    random_state=42,
    stratify=y_eff
)

Logistic Regression

In [6]:
log_pattern = LogisticRegression(max_iter=2000)
log_pattern.fit(X_train, y_pattern_train)

log_pattern_acc = log_pattern.score(X_test, y_pattern_test)
print("Logistic Regression Pattern Accuracy:", log_pattern_acc)

Logistic Regression Pattern Accuracy: 0.9333333333333333


Random Forest

In [7]:
rf_pattern = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_pattern.fit(X_train, y_pattern_train)

rf_pattern_acc = rf_pattern.score(X_test, y_pattern_test)
print("Random Forest Pattern Accuracy:", rf_pattern_acc)

Random Forest Pattern Accuracy: 0.925


Compare

In [8]:
best_pattern_model = rf_pattern  # or log_pattern

y_pred_pattern = best_pattern_model.predict(X_test)

print(confusion_matrix(y_pattern_test, y_pred_pattern))
print(classification_report(
    y_pattern_test,
    y_pred_pattern,
    target_names=pattern_encoder.classes_
))

[[19  0  1  0  0  0]
 [ 1 17  0  2  0  0]
 [ 0  0 20  0  0  0]
 [ 0  0  0 20  0  0]
 [ 0  0  1  0 19  0]
 [ 1  1  2  0  0 16]]
                     precision    recall  f1-score   support

        brute_force       0.90      0.95      0.93        20
dynamic_programming       0.94      0.85      0.89        20
            hashing       0.83      1.00      0.91        20
          recursion       0.91      1.00      0.95        20
              stack       1.00      0.95      0.97        20
        two_pointer       1.00      0.80      0.89        20

           accuracy                           0.93       120
          macro avg       0.93      0.92      0.92       120
       weighted avg       0.93      0.93      0.92       120



Logistic Regression(Efficiency)

In [9]:
log_eff = LogisticRegression(max_iter=2000)
log_eff.fit(X_train, y_eff_train)

log_eff_acc = log_eff.score(X_test, y_eff_test)
print("Logistic Regression Efficiency Accuracy:", log_eff_acc)

Logistic Regression Efficiency Accuracy: 0.9333333333333333


Random Forest (Efficiency)

In [10]:
rf_eff = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_eff.fit(X_train, y_eff_train)

rf_eff_acc = rf_eff.score(X_test, y_eff_test)
print("Random Forest Efficiency Accuracy:", rf_eff_acc)

Random Forest Efficiency Accuracy: 0.9083333333333333


Efficiency Report

In [11]:
best_eff_model = rf_eff  # or log_eff

y_pred_eff = best_eff_model.predict(X_test)

print(confusion_matrix(y_eff_test, y_pred_eff))
print(classification_report(
    y_eff_test,
    y_pred_eff,
    target_names=eff_encoder.classes_
))

[[35  4  0]
 [ 1 38  1]
 [ 2  3 36]]
              precision    recall  f1-score   support

   efficient       0.92      0.90      0.91        39
 inefficient       0.84      0.95      0.89        40
  suboptimal       0.97      0.88      0.92        41

    accuracy                           0.91       120
   macro avg       0.91      0.91      0.91       120
weighted avg       0.91      0.91      0.91       120



Choose Final Models

In [12]:
final_pattern_model = best_pattern_model
final_eff_model = best_eff_model

Train Final Models On FULL Dataset

In [13]:
final_pattern_model.fit(X, y_pattern)
final_eff_model.fit(X, y_eff)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


Save Models

In [14]:
import os
import joblib

# Ensure models folder exists
os.makedirs("backend/models", exist_ok=True)

# Retrain on full dataset
final_pattern_model.fit(X, y_pattern)
final_eff_model.fit(X, y_eff)

# Save models
joblib.dump(final_pattern_model, "backend/models/pattern_model.pkl")
joblib.dump(pattern_encoder, "backend/models/pattern_label_encoder.pkl")

joblib.dump(final_eff_model, "backend/models/efficiency_model.pkl")
joblib.dump(eff_encoder, "backend/models/efficiency_label_encoder.pkl")

print("Final models saved successfully.")

Final models saved successfully.
